# SynthText scope30-v1: generation, training and analysis walkthrough

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kausthubabelagodbdes24-oss/LLM-synthetic-data-classifiers/blob/main/notebooks/SynthText_scope30_walkthrough.ipynb)

This notebook is a presentation-friendly technical walkthrough. It is **not** the professor-facing slide deck. The final slides and submission logistics are deliberately kept outside GitHub under ADR 0014.

## Two evidence lanes — do not mix them

1. **FAKE DEMONSTRATION — NOT EXPERIMENTAL EVIDENCE.** Committed sample fixtures exercise zero-shot, few-shot and AugGPT generation, hashing-embedder training and analysis without Ollama or private data.
2. **ACCEPTED REAL RESULTS — READ ONLY.** The notebook reads the reviewed, committed scope30-v1 analysis. It never regenerates frozen corpora, retrains the accepted grid or rewrites accepted outputs.

The accepted generator was local `qwen3:4b` with a pinned digest and Ollama. Colab does not contain the private real data or approved local model identity, so it must not pretend to reproduce that run or substitute a different generator.

In [ ]:
from pathlib import Path
import hashlib, json, os, subprocess, sys

REPO_URL = 'https://github.com/kausthubabelagodbdes24-oss/LLM-synthetic-data-classifiers.git'
REPO_REF = 'main'

root = Path.cwd()
if not (root / 'config.yaml').is_file():
    root = Path('/content/LLM-synthetic-data-classifiers')
    if not (root / '.git').exists():
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(root)], check=True)
    if REPO_REF != 'main':
        subprocess.run(['git', '-C', str(root), 'fetch', '--depth', '1', 'origin', REPO_REF], check=True)
        subprocess.run(['git', '-C', str(root), 'checkout', '--detach', 'FETCH_HEAD'], check=True)
os.chdir(root)
ROOT = Path.cwd()
REVISION = subprocess.run(['git', 'rev-parse', 'HEAD'], check=True, capture_output=True, text=True).stdout.strip()
print('Repository:', ROOT)
print('Notebook checkout:', REVISION)

In [ ]:
# Lightweight dependencies for the safe fake walkthrough and committed-result tables.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'scikit-learn>=1.4', 'pandas>=2.1', 'matplotlib>=3.8',
                'pyyaml>=6.0', 'requests>=2.31'], check=True)
import pandas as pd
try:
    from IPython.display import Image, Markdown, display
except ImportError:  # Colab provides IPython; this fallback keeps the walkthrough testable elsewhere.
    Markdown = str
    Image = lambda filename: filename
    display = print
print('Walkthrough dependencies ready.')

## 1. Experimental design

Two tasks (TREC and Emotion) × six conditions × seed 13 produce exactly twelve result keys. C1–C4 have equal classifier input counts, but not equal real-label costs. C5a and C5b intentionally differ in training size.

In [ ]:
conditions = pd.DataFrame([
    ('C1', '30 real examples/label', 180, 'real training rows'),
    ('C2', '30 zero-shot synthetic/label', 180, 'no real demonstrations'),
    ('C3', '30 few-shot synthetic/label', 180, 'real demonstrations disclosed separately'),
    ('C4', '15 real + 15 few-shot/label', 180, '90 real + 90 synthetic'),
    ('C5a', '10 real sources/label', 60, 'exact low-resource sources'),
    ('C5b', 'same sources + 6 paraphrases/source', 420, '60 real + 360 AugGPT'),
], columns=['condition', 'construction', 'rows_per_task', 'cost_note'])
display(conditions)

## 2. What the accepted real workflow did

The commands below are shown for process transparency only. **This notebook does not execute them on real data.** The real workflow required local Ollama, the approved Qwen digest, private canonical data, reviewed promotion/freeze evidence, the pinned MiniLM snapshot and clean checkouts.

In [ ]:
real_workflow = pd.DataFrame([
    (1, 'Prepare/verify canonical real splits', 'python -m ws2_training.prepare_data --verify'),
    (2, 'Generate zero-shot, few-shot and AugGPT artifacts', 'ws1_generation.generate with approved local Ollama identity'),
    (3, 'Validate, review and freeze six accepted artifacts', 'python -m ws1_generation.freeze_scope30 --verify'),
    (4, 'Check all twelve training sets', 'python -m ws2_training.train_scope30 --check-inputs'),
    (5, 'Train once and validate separately', 'python -m ws2_training.train_scope30 ... then --validate'),
    (6, 'Analyse the accepted frozen grid', 'python -m ws3_evaluation.analyze_scope30 --check ...'),
], columns=['step', 'purpose', 'recorded_command_or_interface'])
display(real_workflow)

## 3. Safe live demonstration: fake generation and training

The next cell runs only committed fake fixtures. It uses the repository's fake generator and hashing embedder, writes only ignored `data/sample/synthetic-fixtures/` and `results/sample/` paths, and never calls Ollama. Set `RUN_FAKE_PIPELINE = False` when you only want to show the commands.

In [ ]:
RUN_FAKE_PIPELINE = True
fake_env = os.environ.copy()
fake_env.update({
    'SYNTHTEXT_DATA': 'data/sample',
    'SYNTHTEXT_RESULTS': 'results/sample',
    'SYNTHTEXT_EMBEDDER': 'hashing',
    'SYNTHTEXT_GENERATION_MODE': 'fake',
    'PYTHONUTF8': '1',
})
fake_commands = [
    [sys.executable, '-m', 'ws1_generation.generate', '--setting', 'zero_shot', '--dry-run'],
    [sys.executable, '-m', 'ws1_generation.generate', '--setting', 'few_shot', '--dry-run'],
    [sys.executable, '-m', 'ws1_generation.generate', '--setting', 'auggpt', '--dry-run'],
    [sys.executable, '-m', 'ws2_training.train'],
    [sys.executable, '-m', 'ws3_evaluation.analyze'],
]
print('FAKE DEMONSTRATION — NOT EXPERIMENTAL EVIDENCE')
for command in fake_commands:
    print(' '.join(command))
    if RUN_FAKE_PIPELINE:
        completed = subprocess.run(command, cwd=ROOT, env=fake_env, check=True, capture_output=True, text=True)
        print('  completed; verbose fake logs suppressed — use the labelled previews below')
print('Fake walkthrough complete.' if RUN_FAKE_PIPELINE else 'Commands shown only; nothing executed.')

In [ ]:
# Inspect a few fake generated rows. Their text and scores have no scientific meaning.
fake_files = sorted((ROOT / 'data/sample/synthetic-fixtures').glob('**/*.jsonl'))
preview = []
for path in fake_files:
    for line in path.read_text(encoding='utf-8').splitlines()[:2]:
        row = json.loads(line)
        preview.append({'file': path.name, 'id': row.get('id'), 'label': row.get('label'), 'text': row.get('text')})
display(Markdown('**FAKE DEMONSTRATION — NOT EXPERIMENTAL EVIDENCE**'))
display(pd.DataFrame(preview).head(12) if preview else Markdown('Run the fake pipeline cell to create fixture rows.'))

In [ ]:
fake_runs = ROOT / 'results/sample/runs.csv'
display(Markdown('**FAKE DEMONSTRATION — NOT EXPERIMENTAL EVIDENCE**'))
if fake_runs.is_file():
    display(pd.read_csv(fake_runs))
else:
    display(Markdown('Run the fake pipeline cell to create fixture training rows.'))

## 4. Accepted real results — read only

From here onward, the notebook reads the four lead-reviewed files committed by issue #14. It verifies their recorded hashes before displaying them. No training or generation occurs.

In [ ]:
ANALYSIS_DIR = ROOT / 'docs/analysis/scope30-v1-seed13'
analysis = json.loads((ANALYSIS_DIR / 'analysis.json').read_text(encoding='utf-8'))
for name, expected in analysis['outputs_sha256'].items():
    actual = hashlib.sha256((ANALYSIS_DIR / name).read_bytes()).hexdigest()
    assert actual == expected, f'{name}: expected {expected}, got {actual}'
assert analysis['generation_mode'] == 'real' and analysis['real_run_evidence'] is True
print('Accepted files verified.')
print('Grid revision:', analysis['grid']['revision'])
print('grid_complete.json:', analysis['grid']['grid_complete_sha256'])
print('Analysis revision:', analysis['analysis_code']['revision'])
print('Uncertainty:', analysis['uncertainty'])

In [ ]:
real_results = pd.DataFrame(analysis['results'])
real_results = real_results[['task', 'condition', 'n_train', 'macro_f1', 'accuracy']]
display(real_results.style.format({'macro_f1': '{:.4f}', 'accuracy': '{:.4f}'}))

In [ ]:
gap_table = pd.DataFrame(analysis['gaps'])
display(gap_table[['task', 'condition', 'c1_macro_f1', 'macro_f1', 'absolute_gap', 'relative_gap', 'status']]
        .style.format({'c1_macro_f1': '{:.4f}', 'macro_f1': '{:.4f}', 'absolute_gap': '{:+.4f}',
                       'relative_gap': lambda x: 'undefined' if pd.isna(x) else f'{x:+.2%}'}))

In [ ]:
display(Image(filename=str(ANALYSIS_DIR / 'f1_by_condition.png')))
display(Image(filename=str(ANALYSIS_DIR / 'gap.png')))

In [ ]:
costs = pd.DataFrame(analysis['costs'])
display(costs[['task', 'condition', 'n_train', 'real_training_rows', 'synthetic_training_rows',
               'real_generation_demonstrations', 'note']])

## 5. How to interpret the accepted results

- TREC C2/C3 were above same-task C1 on this run; Emotion C2/C3 were below C1. This is consistent with the planned direction but does **not** show subjectivity caused the task difference.
- Few-shot versus zero-shot, C4 versus C1 and C5b versus C5a are descriptive single-run comparisons. Their reliability is unknown with one seed.
- C5b has 420 rows versus C5a's 60, so its comparison mixes content and training-size effects.
- Demonstration costs, model-assisted quality flags and the disclosed TREC train/test duplicate remain part of the limitations.
- There is no standard deviation, confidence interval, significance test, stability estimate or human-agreement result.

For the complete reviewed wording, open [`summary.md`](../docs/analysis/scope30-v1-seed13/summary.md) and [`scope30-v1-seed13-interpretation.md`](../docs/analysis/scope30-v1-seed13-interpretation.md).

## Takeaway

This notebook makes the engineering workflow easy to demonstrate while preserving the evidence boundary: fake fixtures show *how the code works*; committed accepted artifacts show *what the real experiment observed*. The final professor-facing presentation remains a separate owner-prepared deliverable outside GitHub.